# Grouping and aggregating

Python stage, lesson P14. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** sort a table with `sort_values` and see where missing values go; explain `groupby` as split, apply and combine; compute several named aggregates in one pass; tell `size` from `count`, and check that the group totals add up to the table's total.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import numpy as np
import pandas as pd

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")
# The order lines with their price and amount, as built in lesson P13.
price_of = dict(zip(products["product_id"], products["unit_price_usd"]))
lines = order_items.assign(price=order_items["product_id"].map(price_of))
lines = lines.assign(amount=lines["quantity"] * lines["price"] - lines["discount_usd"])

## Unit 1: sort_values, and where the blanks go

**Predict first.** which three products come first, and does `products` itself change its order?

In [ ]:
priciest = products.sort_values("unit_price_usd", ascending=False)
print(priciest[["product_name", "unit_price_usd"]].head(3))
print(products["product_id"].head(3).tolist(), priciest["product_id"].head(3).tolist())

**Predict first.** sorted from the highest, are the first three values numbers or blanks, and the last three?

In [ ]:
by_referrer = customers.sort_values("referred_by", ascending=False)
print(by_referrer["referred_by"].head(3).tolist())
print(by_referrer["referred_by"].tail(3).tolist())

## Unit 2: groupby: split, apply, combine

**Predict first.** how many rows will `per_store` have, and will its numbers add up to the rows of `orders`?

In [ ]:
per_store = orders.groupby("store_id").size()
print(per_store)
print(per_store.sum(), len(orders))

**Predict first.** the walk-in order has a blank `customer_id`. How many groups will there be, and will their sizes add up to 1,800?

In [ ]:
per_customer = orders.groupby("customer_id").size()
print(len(per_customer), per_customer.sum())

## Unit 3: agg: several named results in one pass

**Predict first.** how many rows and columns will `by_product` have? The `products` table has 16 products.

In [ ]:
by_product = lines.groupby("product_id").agg(
    units=("quantity", "sum"),
    sales=("amount", "sum"),
    n_lines=("order_item_id", "size"),
)
print(by_product.shape)
print(by_product.head(3))

**Predict first.** which three product ids come first when the products are sorted by sales, highest first?

In [ ]:
ranked = by_product.sort_values("sales", ascending=False)
print(ranked.head(3))

## Unit 4: size counts rows, count counts values

**Predict first.** the walk-in order has no customer id. For which payment method will the two numbers differ, and by how much?

In [ ]:
by_method = orders.groupby("payment_method").agg(
    orders=("order_id", "size"),
    with_customer=("customer_id", "count"),
)
print(by_method)
print(by_method["orders"] - by_method["with_customer"])

## Unit 5: Check: the groups add up

**Predict first.** do the totals of the 15 groups equal the totals of the whole table?

In [ ]:
print(by_product["sales"].sum(), lines["amount"].sum())
print(by_product["n_lines"].sum(), len(lines))

## Finish it

### Exercise 1: Finish it

**Task:** What share of the 150 customers were brought in by the ten busiest referrers? `referrer_counts` counts the customers that each referrer brought in with `value_counts()`. Replace the first `None` with the same counts made with `groupby` and `size`, then set `answer_1` to the share, a number between 0 and 1.

**Expected:** one decimal number between 0 and 1.

In [ ]:
referrer_counts = customers["referred_by"].value_counts()  # given: the customers each referrer brought in
sizes = None  # replace None: the same counts made with groupby and size
answer_1 = None  # then set answer_1 to the customers brought in by the 10 busiest referrers, divided by the 150 customers of the table

In [ ]:
_ = check("Exercise 1", "answer_1", 11 / 150, traps={11 / 30: "You divided by the sum of the group sizes, 30. The customers without a referrer are in no group, so the groups add up to less than the table. Divide by the rows of customers."})

<details>
<summary>Hint</summary>

A groupby of one column with `size()` gives the same counts as `value_counts()`. Unit 2 shows what the sizes add up to.

</details>

<details>
<summary>Solution</summary>

```python
sizes = customers.groupby("referred_by").size()
ten_busiest = sizes.sort_values(ascending=False).head(10)
answer_1 = ten_busiest.sum() / len(customers)
```

The answer is 0.0733: the ten busiest referrers brought in 11 of the 150 customers. `value_counts()` and `groupby(...).size()` hold the same counts, and only the order of their rows can differ. Dividing by `sizes.sum()`, 30, gives 0.3667: the 120 customers without a referrer are in no group, as the walk-in order was in unit 2, and the check cell says so.

</details>

## On your own

### Exercise 2: On your own

**Task:** For each city of the customers, what share of the customers were referred by another customer? Report the highest share. Before you start, say in one sentence what kind of number the answer is. Make one table with `groupby` and `agg`: the customers of each city and the referred customers of each city. The answer is the highest share. Write the code in the empty cell below and set `answer_2`.

**Expected:** one decimal number between 0 and 1.

In [ ]:
_ = check("Exercise 2", "answer_2", 15 / 46, traps={1.0: "A share of exactly 1 means every customer counted as referred. size counts the rows of a group; count skips the blanks of referred_by."})

<details>
<summary>Hint</summary>

Unit 4 counts the rows of a group with one function and the values that are not blank with another. A customer who was not referred has a blank `referred_by`.

</details>

<details>
<summary>Solution</summary>

```python
by_city = customers.groupby("city").agg(
    customers=("customer_id", "size"),
    referred=("referred_by", "count"),
)
answer_2 = (by_city["referred"] / by_city["customers"]).max()
```

The answer is 0.3261: Berkeley, where 15 of the 46 customers were referred. Counting `referred_by` with `size` counts every customer, so every share is exactly 1, and the check cell says so.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague lists the three products that sold the least, and wants the Pumpkin Spice Latte, product 16, on the list: it was never sold, so its sales are blank. The list does not show it. Fix the line, then set `answer_3` to the product ids of the first three rows, as a list. `by_product` is the table of unit 3.

**Expected:** a list of three product ids.

In [ ]:
menu_sales = products.assign(sales=products["product_id"].map(by_product["sales"]))
least = menu_sales.sort_values("sales")  # the colleague's line
answer_3 = None  # fix the line above, then set answer_3 = least.head(3)["product_id"].tolist()

In [ ]:
_ = check("Exercise 3", "answer_3", [16, 7, 9], traps={(7, 9, 1): "Product 16 is not on the list: its sales are blank, and blanks sort last. Which parameter of sort_values decides where they go?"})

<details>
<summary>Hint</summary>

Print `menu_sales.tail(2)`. Where did the product with a blank sales value go? The pandas page for `sort_values` names the parameter that decides it.

</details>

<details>
<summary>Solution</summary>

```python
menu_sales = products.assign(sales=products["product_id"].map(by_product["sales"]))
least = menu_sales.sort_values("sales", na_position="first")
answer_3 = least.head(3)["product_id"].tolist()
```

The answer is `[16, 7, 9]`: the Pumpkin Spice Latte, which never sold, then Green Tea and Iced Lemon Tea. `map` gave product 16 a blank, and the colleague's sort put the blank last, so the list read `[7, 9, 1]` and missed it; the check cell says so.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")